# HTTP clients (httpx, requests), retries with tenacity

*0.1 Python for GenAI · run **Setup** first*

## Setup

Settings, a configured client, and two helpers. Every cell below uses them.

In [1]:
"""Shared setup for this notebook: typed settings, configured clients, logging."""

import asyncio
import json
import logging
from concurrent.futures import ThreadPoolExecutor

from dotenv import find_dotenv
from openai import AsyncOpenAI, OpenAI
from pydantic import Field, SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict


class Settings(BaseSettings):
    """All configuration in one validated object, read from the environment / .env."""

    model_config = SettingsConfigDict(env_file=find_dotenv(), extra="ignore")

    openai_api_key: SecretStr
    openai_model: str = "gpt-4o-mini"
    request_timeout_seconds: float = Field(default=30, gt=0)
    max_retries: int = Field(default=2, ge=0, le=5)


settings = Settings()

client = OpenAI(
    api_key=settings.openai_api_key.get_secret_value(),
    timeout=settings.request_timeout_seconds,
    max_retries=settings.max_retries,
)


def async_client() -> AsyncOpenAI:
    """A fresh async client per event loop (async clients are bound to the loop they run in)."""
    return AsyncOpenAI(
        api_key=settings.openai_api_key.get_secret_value(),
        timeout=settings.request_timeout_seconds,
        max_retries=settings.max_retries,
    )


def run_async(coroutine):
    """Run a coroutine from a notebook (which already has an event loop). Scripts use asyncio.run()."""
    with ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(asyncio.run, coroutine).result()


def show(title: str, value) -> None:
    """Print a labelled, formatted JSON block."""
    print(title)
    print(json.dumps(value, indent=2, ensure_ascii=False, default=str))


logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
for noisy in ["httpx", "httpx2", "httpcore", "openai"]:
    logging.getLogger(noisy).setLevel(logging.WARNING)

print("model:", settings.openai_model, "| timeout:", settings.request_timeout_seconds, "s")

model: gpt-4o-mini | timeout: 30.0 s


### httpx

> **Problem.** The team adopts an internal model server that speaks a slightly different protocol; no SDK exists. And during an incident nobody can say what the OpenAI SDK actually sends — which headers, which timeout, how many retries — because it was always a black box.

**Idea.** One POST with headers and a JSON body; a JSON reply with a status code. Everything else is convenience.

**Use when** no SDK, internal servers, custom transports.  
**Not when** an official SDK exists — it handles retries, streaming and error types.

```mermaid
flowchart LR
    C[your code] -->|"POST /chat/completions · Authorization · JSON body"| P[provider]
    P -->|"200 + JSON · x-ratelimit-remaining"| C
```

**How it works.**
1. `httpx.Client(base_url=..., headers=..., timeout=..., transport=...)` is built once: it keeps a pool of open connections and applies the same settings to every call.
2. `http.post("/chat/completions", json=body)` sends a POST with the bearer token header and the JSON body — exactly what the SDK sends.
3. `response.raise_for_status()` turns any 4xx/5xx status into an exception; otherwise `response.json()` is the reply as a dictionary.
4. Reply headers carry the rate-limit budget (`x-ratelimit-remaining-requests`) — the SDK reads these too.
5. A second call with a bad model name shows the failure path: status 404 and the provider's error message in the body.

| | what happens | result |
|:--|:--|:--|
| ✓ | valid model | 200, "pong", usage counts, rate-limit headers |
| ✗ | `no-such-model` | 404 with the provider's message |

**Production code and its real output**

In [2]:
# httpx — a production HTTP client: pooled connections, explicit connect/read/write timeouts,
# transport-level retries, bearer auth. This is the wire call every SDK wraps.
import httpx

http = httpx.Client(
    base_url="https://api.openai.com/v1",
    headers={"Authorization": f"Bearer {settings.openai_api_key.get_secret_value()}"},
    timeout=httpx.Timeout(connect=5.0, read=30.0, write=10.0, pool=5.0),
    transport=httpx.HTTPTransport(retries=2),
)
body = {
    "model": settings.openai_model,
    "messages": [{"role": "user", "content": "Reply with: pong"}],
    "max_tokens": 3,
}

with http:
    response = http.post("/chat/completions", json=body)
    response.raise_for_status()
    print(
        "status:",
        response.status_code,
        "| reply:",
        response.json()["choices"][0]["message"]["content"],
    )
    print("rate limit remaining:", response.headers.get("x-ratelimit-remaining-requests"))

    try:
        http.post("/chat/completions", json={**body, "model": "no-such-model"}).raise_for_status()
    except httpx.HTTPStatusError as error:
        print(
            "bad model ->",
            error.response.status_code,
            error.response.json()["error"]["message"][:50],
        )
        bad_status = error.response.status_code
assert bad_status == 404

status: 200 | reply: pong
rate limit remaining: 9890
bad model -> 404 The model `no-such-model` does not exist or you do


**What the output shows.** Status 200 with the reply, token usage and the remaining rate-limit budget; the bad model name returned 404 with the provider's explanation.

**In practice**
- **one client** — a new client per request opens a new TLS connection each time: ~300 ms instead of ~50 ms, and eventually socket exhaustion.
- **four timeouts** — the default is 5 s for everything, which kills long generations; set connect, read, write and pool explicitly.
- **map errors** — `raise_for_status()` then translate: 429 → retry with backoff, 401 → stop and alert, 5xx → retry, 400 → fix the request.
- **read the headers** — rate-limit and request-id headers are how you debug throttling and how you configure limiters.
- **close it** — close the client on shutdown (or use `with`) so connections are released.

**Alternatives** — the vendor SDK (retries, streaming, typed errors for free) · requests (sync only) · aiohttp

**Terms** — *status code*: 200 ok · 404 not found · 429 slow down · 500 server error · *header*: extra info with a request: your key, rate limits · *connection pool*: reused open connections


### requests

> **Problem.** A nightly job built on `requests` hangs at 3 a.m. and is still hanging at 9. The provider had a network blip; the bare `requests.post()` had no timeout, so it waited forever, and the job's lock stopped every later run.

**Idea.** Same call, made safe: a Session, a retry adapter, a timeout on every call.

**Use when** maintaining existing code.  
**Not when** new async services — use httpx.

```
requests.post(url, json=…)                    ──▶ no timeout · no retries
session.post(url, json=…, timeout=(5, 30))    ──▶ bounded · adapter retries 429/5xx
```

**How it works.**
1. `requests.Session()` reuses connections and holds default headers (the bearer token).
2. `HTTPAdapter(max_retries=Retry(total=3, backoff_factor=0.5, status_forcelist=[429, 500, 502, 503, 504]))` mounted on the session retries those statuses with growing waits.
3. Every call passes `timeout=(5, 30)`: 5 s to connect, 30 s to read — the one argument `requests` will never supply for you.
4. `raise_for_status()` converts error statuses into exceptions, as with httpx.
5. A second call with `timeout=0.001` shows the failure you *want*: a `ConnectTimeout` after a millisecond, not a hang.

| | what happens | result |
|:--|:--|:--|
| ✓ | `timeout=(5, 30)` | 200, "pong" |
| ✗ | `timeout=0.001` | ConnectTimeout — and you *see* it instead of hanging |

**Production code and its real output**

In [3]:
# requests — same call, configured the way it must be: a Session with a retry adapter (backoff on
# 429/5xx) and a timeout on every call. Never call requests.post() bare in a service.
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

session = requests.Session()
session.mount(
    "https://",
    HTTPAdapter(
        max_retries=Retry(
            total=3,
            backoff_factor=0.5,
            status_forcelist=[429, 500, 502, 503, 504],
            allowed_methods=["POST"],
        )
    ),
)
session.headers["Authorization"] = f"Bearer {settings.openai_api_key.get_secret_value()}"
body = {
    "model": settings.openai_model,
    "messages": [{"role": "user", "content": "Reply with: pong"}],
    "max_tokens": 3,
}

response = session.post("https://api.openai.com/v1/chat/completions", json=body, timeout=(5, 30))
response.raise_for_status()
print("reply:", response.json()["choices"][0]["message"]["content"])

try:
    session.post("https://api.openai.com/v1/chat/completions", json=body, timeout=0.001)
except requests.exceptions.RequestException as error:
    print("timeout=0.001 ->", type(error).__name__)
    timeout_error = type(error).__name__
assert "Timeout" in timeout_error or "Connect" in timeout_error

WARNING urllib3.connectionpool: Retrying (Retry(total=2, connect=None, read=None, redirect=None, status=None)) after connection broken by 'ReadTimeoutError("HTTPSConnectionPool(host='api.openai.com', port=443): Read timed out. (read timeout=0.001)")': /v1/chat/completions


reply: Pong!


WARNING urllib3.connectionpool: Retrying (Retry(total=1, connect=None, read=None, redirect=None, status=None)) after connection broken by 'ConnectTimeoutError(<HTTPSConnection(host='api.openai.com', port=443) at 0x108af2570>, 'Connection to api.openai.com timed out. (connect timeout=0.001)')': /v1/chat/completions


WARNING urllib3.connectionpool: Retrying (Retry(total=0, connect=None, read=None, redirect=None, status=None)) after connection broken by 'ConnectTimeoutError(<HTTPSConnection(host='api.openai.com', port=443) at 0x108af2810>, 'Connection to api.openai.com timed out. (connect timeout=0.001)')': /v1/chat/completions


timeout=0.001 -> ConnectTimeout


**What the output shows.** The session call succeeded; the deliberately tiny timeout failed immediately with a named exception — the behaviour a missing timeout silently denies you.

**In practice**
- **timeout on every call** — there is no client-level default in `requests`; a code review rule, or a wrapper function, is the only protection.
- **retry adapter** — put backoff on the session so every call gets it; do not hand-roll retry loops around each `post`.
- **idempotency** — the adapter retries POSTs only if you allow it (`allowed_methods`); make sure the server can deduplicate.
- **migrate** — new services use httpx (async, HTTP/2, per-client timeouts); keep `requests` for code you inherit.

**Alternatives** — httpx · urllib3 directly

**Terms** — *Session*: reuses connections and holds default headers · *adapter*: a plug-in that adds retries with backoff


### retries with tenacity

> **Problem.** One request in two hundred fails with a 429 or a dropped connection. Each one becomes an error in the user's chat window and a support ticket, even though the same request a second later would have worked. Meanwhile a wrong API key produces the *same* error handling: three slow retries before anyone is told the key is bad.

**Idea.** Retry automatically — only for failures that can go away by themselves.

**Use when** calling anything that fails briefly: providers, databases, queues.  
**Not when** the error is permanent: wrong key, bad request.

```mermaid
flowchart LR
    A[request] --> B{reply}
    B -->|ok| C[done]
    B -->|"429 · 5xx · timeout"| D["wait 0.3 → 0.6 → 1.2 s + jitter"] --> A
    B -->|"401 · 400"| E[fail now]
    D -->|4th failure| F[give up]
```

**How it works.**
1. `is_transient(error)` decides what is worth retrying: timeouts, connection errors, 429 and 5xx. Everything else — 401, 400 — is permanent.
2. `@retry(retry=retry_if_exception(is_transient), ...)` wraps `post_completion`; on a transient error tenacity calls it again.
3. `wait_random_exponential(multiplier=0.3, max=5)` waits ~0.3 s, then ~0.6, ~1.2 … with a random component, capped at 5 s.
4. `stop_after_attempt(4)` gives up after the fourth try and re-raises the last error (`reraise=True`), so the caller sees the real exception.
5. `before_sleep_log` writes one warning line per retry — the audit trail you read when a provider has a bad hour.

| | what happens | result |
|:--|:--|:--|
| ✗ no retry | 429 → error shown to user | 1 in 200 requests fail |
| ✓ retry | 429 → wait 0.3 s → 200 "pong" | 0 fail, +0.3 s on 1 in 200 |
| ✓ still stop | 401 → fail now | waiting cannot fix a wrong key |

**Production code and its real output**

In [4]:
# tenacity — the retry policy a service defines once: retry only transient failures (timeouts,
# 429, 5xx), exponential backoff with jitter, a hard cap, a log line before each sleep.
# Permanent failures (401) fail immediately.
import time

import httpx
from tenacity import (
    before_sleep_log,
    retry,
    retry_if_exception,
    stop_after_attempt,
    wait_random_exponential,
)

log = logging.getLogger("retry")


def is_transient(error: BaseException) -> bool:
    if isinstance(error, (httpx.TimeoutException, httpx.ConnectError)):
        return True
    return isinstance(error, httpx.HTTPStatusError) and error.response.status_code in (
        429,
        500,
        502,
        503,
        504,
    )


attempts: list[float] = []


@retry(
    retry=retry_if_exception(is_transient),
    wait=wait_random_exponential(multiplier=0.3, max=5),
    stop=stop_after_attempt(4),
    before_sleep=before_sleep_log(log, logging.WARNING),
    reraise=True,
)
def post_completion(timeout: float, api_key: str) -> dict:
    attempts.append(time.perf_counter())
    with httpx.Client(timeout=timeout) as http:
        response = http.post(
            "https://api.openai.com/v1/chat/completions",
            headers={"Authorization": f"Bearer {api_key}"},
            json={
                "model": settings.openai_model,
                "messages": [{"role": "user", "content": "Reply with: pong"}],
                "max_tokens": 3,
            },
        )
    response.raise_for_status()
    return response.json()


key = settings.openai_api_key.get_secret_value()
attempts.clear()
print(
    "healthy:",
    post_completion(30, key)["choices"][0]["message"]["content"],
    "| attempts",
    len(attempts),
)

attempts.clear()
try:
    post_completion(0.001, key)
except httpx.TimeoutException:
    print("1 ms timeout: attempts", len(attempts), "(transient -> retried)")
    timeout_attempts = len(attempts)

attempts.clear()
try:
    post_completion(30, "sk-invalid")
except httpx.HTTPStatusError as error:
    print(
        "bad key:",
        error.response.status_code,
        "| attempts",
        len(attempts),
        "(permanent -> not retried)",
    )
    auth_attempts = len(attempts)
assert timeout_attempts == 4 and auth_attempts == 1

WARNING retry: Retrying __main__.post_completion in 0.0693 seconds as it raised ConnectTimeout: timed out.


WARNING retry: Retrying __main__.post_completion in 0.321 seconds as it raised ConnectTimeout: timed out.


healthy: pong | attempts 1


WARNING retry: Retrying __main__.post_completion in 0.854 seconds as it raised ConnectTimeout: timed out.


1 ms timeout: attempts 4 (transient -> retried)


bad key: 401 | attempts 1 (permanent -> not retried)


**What the output shows.** The healthy call took one attempt. The 1 ms timeout was retried four times with growing waits (logged), then gave up. The bad key failed on the first attempt with no retries.

**In practice**
- **jitter** — without randomness, 1,000 clients that failed together retry together — three synchronised waves that turn a blip into an outage.
- **one layer** — the SDK retries twice by default; add tenacity on top and a caller loop above that and one failure costs 27 attempts. Retry in one place, deliberately.
- **honour Retry-After** — when the provider says how long to wait, use that instead of your own backoff.
- **idempotency** — a retried POST can create a second order or send a second email; pass an idempotency key so the server deduplicates.
- **circuit breaker** — retries help with blips; during a real outage they add load. Pair them with a breaker that stops calling a dead backend for a while.

**Alternatives** — the SDK's `max_retries` (simplest, covers the common cases) · a gateway retry policy (one place for all services)

**Terms** — *429*: provider's "slow down" reply · *jitter*: small random extra wait · *backoff*: waiting longer each time
